# Experiment 4: Latency Tradeoff

Inspect the benchmark of eight questions: two per populated retrieval subgroup. Document embedding and index preparation are outside the timed paths.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.


**Reference input scope:** all four experiments select from the frozen **15,585 fully covered question IDs**. Targeted reranking uses **500 opportunities + 200 controls**; answer quality samples **up to 25 per resulting subgroup**, and latency samples from those answers. Passage embeddings are reused; live question embedding, reranking, generation and judging calls produce outputs in this notebook.

**Workflow:** Experiment 1 (Baseline) --> Diagnosis --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> **this Experiment 4 (Latency Tradeoff)**


**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## 1. Set up the latency study

**What:** prepare the end-to-end benchmark and standard scoring.

**Why:** extend the v3 answer-quality story with the operational cost of a complete reranked path.

Use the root pinned `requirements.txt`. Live execution requires `OPENROUTER_API_KEY`, prepared document vectors and the FAISS index. Model settings and portable artifact paths live in `config.json`. Optional new benchmarks and plot exports use `requirements-latency.txt`.

In [1]:
STUDY = 'latency'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,8
Generation limit,400 tokens


## 2. Keep the frozen sample and run order

The frozen benchmark sample has **eight questions**, sampled with seed 43: two each from opportunity gain, unchanged, loss, and all-support controls. The fifth planned subgroup has no eligible cases; it is not pending work.

The source answer-quality sample has 79 questions. Sampled IDs precede benchmark generation and scoring. Arms run sequentially with one worker, alternating which arm runs first by sample position.

These deliberately stratified eight pairs are a descriptive operational check, not a population estimate. Two cases per populated group cannot support stable group-level quality conclusions.

The benchmark contains eight question pairs. The empty fifth subgroup has no eligible cases and is not a pending stage.

In [2]:
view.show('questions')
if MODE != "plan":
    assignments = pd.read_parquet(tutorial.upstream("latency") / "input/questions.parquet")
    display_table(assignments[["query_id", "latency_sample_order", "arm_order"]])


subgroup
control_all_support_already_present    2
opportunity_gain                       2
opportunity_loss                       2
opportunity_unchanged                  2
Name: questions, dtype: int64

query_id,question,subgroup,diagnostic
1046747,what is rx provigil u,control_all_support_already_present,False
106620,cost of repainting cabinets,control_all_support_already_present,False
1062919,what is an exeval,opportunity_gain,False
1084140,what does book mean for utilities,opportunity_gain,False
1079518,what groups benefit from social programs,opportunity_loss,False
1100471,foods that unsaturated,opportunity_loss,False
1083354,what does nodc stand for in state gov,opportunity_unchanged,False
1084423,what does a person on probation have to do,opportunity_unchanged,False


query_id,latency_sample_order,arm_order
1046747,0,"[baseline, reranked]"
106620,1,"[reranked, baseline]"
1062919,2,"[baseline, reranked]"
1084140,3,"[reranked, baseline]"
1079518,4,"[baseline, reranked]"
1100471,5,"[reranked, baseline]"
1083354,6,"[baseline, reranked]"
1084423,7,"[reranked, baseline]"


## 3. Define the complete timed paths

**What:** each path made its own query-embedding request, retrieved 30 passages, selected five and generated an answer. The reranked path added a request to rerank its own exact 30 candidates.

**Why:** measure the operational cost of adding reranking to a full RAG path.

Corpus, precomputed document embeddings, FAISS settings, generator settings and passage budgets were fixed. Document embedding and one-time index/corpus loading were excluded from path timing. A persistent HTTP session served the sequential run.

These are the benchmark’s own answers and contexts. Their earlier v3 counterparts are separate outputs. Each path’s exact request bodies, passages, responses and stage durations remain inspectable.

**What changed from Experiment 3?** Experiment 3 uses frozen top-five contexts. This benchmark made fresh query-embedding and generation requests for each arm, so it measured the complete online path. The eight question IDs are frozen; each path retrieves its own candidates. The reranker sees exactly its path’s retrieved 30 passages. Do not assume both paths or the earlier study have identical candidate sets without comparing their passage IDs.

| Work | Timed? | Why? |
|---|---|---|
| Document embeddings and index/corpus loading | No | Precomputed or loaded before the sequential benchmark. |
| Fresh query embedding and local retrieval | Yes, in both arms | Required to answer each incoming question. |
| Reranking the retrieved 30 passages | Yes, reranked arm only | The extra evidence-selection stage. |
| Select five passages, build prompt and generate | Yes, in both arms | Complete the response. |
| Offline answer scoring | No | Evaluation happens after the response. |


In [3]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))

# Serial online paths, counterbalanced arm order. New HTTP calls and retries are timed.
# Corpus/index loading is outside measured path time. Cached HTTP responses cannot
# be substituted into an incomplete benchmark's timings.
benchmark_run = tutorial.benchmark(embed_api, rerank_api, generate_api)
if benchmark_run is not None:
    display(benchmark_run)
view.show('completion')


benchmark: saved for CURRENT LIVE RUN 20261111T044228Z-219a4a03d428


{'questions': 8,
 'paths': 16,
 'run_id': '20261111T044228Z-219a4a03d428',
 'preparation_excluded_from_timing': True}

Measure,Value
run_id,20261111T044228Z-219a4a03d428
questions,8
successful_answers,16
complete_pairs,8
main_complete_pairs,8
truncated_answers,0
request_outcomes,"{'error': 1, 'ok': 40}"


## 4. Separate timing scopes and check coverage

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

**What:** distinguish successful-request time, successful-path time and total wall time.

**Why:** a retry can dominate the user’s wait even when the eventual successful request is fast.

**Successful path** adds successful HTTP attempt time to local retrieval and prompt selection. **Total wall time** includes all attempts, actual backoff, local work and client overhead. Durations use seconds unless labeled milliseconds.

This **recorded reference run** completed **16 paths (8 pairs)** with **40 successful hosted stages** and **41 timed attempts**. **1 failed attempts** and their actual backoff are retained. **0 answers** reached the token limit. Document preparation is excluded from the timed paths.

| Timing scope | Plain description | What it helps explain |
| --- | --- | --- |
| Successful-request time | Sum of successful hosted attempts across the path. | Hosted work excluding failed attempts. |
| Successful-path time | Successful requests plus local retrieval and prompt selection. | Measured path work excluding failures and backoff. |
| Total end-to-end wall time | All attempts, backoff, local work and client overhead. | Full measured wait for the answer. |

Generation is non-streaming; this measures completion time, not time to first token. A successful-request total is a sum across several stages.

In [4]:
view.show('completion')
view.show('attempts')


Measure,Value
run_id,20261111T044228Z-219a4a03d428
questions,8
successful_answers,16
complete_pairs,8
main_complete_pairs,8
truncated_answers,0
request_outcomes,"{'error': 1, 'ok': 40}"


stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
query_embedding,1083354,baseline,1,ok,200,0.3634,0.0000
stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
generation,1100471,reranked,1,ok,200,2.8335,0.0000
generation,1062919,reranked,1,ok,200,1.9257,0.0000
reranking,1084423,reranked,1,ok,200,0.4801,0.0000
query_embedding,1084423,reranked,1,ok,200,4.6494,0.0000
query_embedding,1100471,baseline,1,ok,200,0.9710,0.0000
reranking,106620,reranked,1,ok,200,0.4119,0.0000
generation,1084140,baseline,1,ok,200,2.5855,0.0000
query_embedding,106620,reranked,1,ok,200,0.3490,0.0000


## 5. Compare complete-path latency

These are completed Qwen measurements for eight paired questions. Compare within-question deltas, retaining failed attempts and actual backoff in total wall time. Network and serving-backend variation are part of these observations; they are not a production latency guarantee. All displayed timings come from this Live run.


In [5]:
view.show('latency')


query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,baseline,20261111T044228Z-219a4a03d428,5.2054,0.8792,0.1373,0.0000,4.1665,4.6810,0.0000,0.0000,4.8182,Unavailable
query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,reranked,20261111T044228Z-219a4a03d428,4.1069,0.3744,0.0388,0.0000,3.2069,4.0139,0.0000,0.0000,4.0527,0.4700
106620,reranked,20261111T044228Z-219a4a03d428,1.9253,0.3642,0.0390,0.0000,1.0644,1.8131,0.0000,0.0000,1.8522,0.4215
106620,baseline,20261111T044228Z-219a4a03d428,2.1702,0.3786,0.0377,0.0000,1.7190,2.0757,0.0000,0.0000,2.1134,Unavailable
1062919,baseline,20261111T044228Z-219a4a03d428,1.9093,0.2789,0.0374,0.0000,1.5628,1.8175,0.0000,0.0000,1.8548,Unavailable
1062919,reranked,20261111T044228Z-219a4a03d428,3.2594,0.8061,0.0385,0.0000,1.9382,3.1469,0.0000,0.0000,3.1853,0.4390
1084140,reranked,20261111T044228Z-219a4a03d428,3.2064,0.3117,0.0350,0.0000,2.2365,3.0996,0.0000,0.0000,3.1346,0.5837
1084140,baseline,20261111T044228Z-219a4a03d428,3.0119,0.3489,0.0359,0.0000,2.5947,2.9201,0.0000,0.0000,2.9561,Unavailable
1079518,baseline,20261111T044228Z-219a4a03d428,2.7525,0.7902,0.0399,0.0000,1.8922,2.6564,0.0000,0.0000,2.6962,Unavailable


total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
3.1846,0.5792,0.0670,0.0000,2.5066,2.9843,0.0000,0.0000,3.0513,Unavailable
5.9251,1.8112,0.0386,0.0000,3.4490,4.4170,0.9976,0.2501,4.4556,0.4498


In [6]:
if MODE != "plan":
    times = pd.read_csv(tutorial.folder / "latency.csv", dtype={"query_id": str})
    paired_wall_time = times.pivot(index="query_id", columns="arm", values="total_end_to_end_seconds").dropna()
    paired_wall_time["reranked_minus_baseline_seconds"] = paired_wall_time.reranked - paired_wall_time.baseline
    display_table(paired_wall_time)


arm,baseline,reranked,reranked_minus_baseline_seconds
,5.2054,4.1069,-1.0985
,1.9093,3.2594,1.3502
,2.1702,1.9253,-0.2449
,2.7525,7.2644,4.5120
,1.4315,12.7325,11.3010
,3.0119,3.2064,0.1945
,3.0751,8.5761,5.5010
,5.9211,6.3300,0.4089


## 6. Identify the stage costs

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

**What:** separate query embedding, local retrieval, reranking, selection and generation.

**Why:** the reranker request’s cost is only one part of the complete path difference.

The hosted reranker stage’s median in this **recorded reference run** is **0.426 seconds**. The paired path difference also includes embedding, generation and local-work variation.

The table labels stage durations in **milliseconds**. The current run’s request records retain exact seconds, attempt counts and actual backoff. Stage medians should not be added to reconstruct a path median.

In [7]:
view.show('latency_stages')


total_end_to_end_seconds_ms,query_embedding_seconds_ms,retrieval_seconds_ms,selection_prompt_seconds_ms,generation_seconds_ms,successful_request_seconds_ms,failed_attempt_seconds_ms,backoff_seconds_ms,successful_path_seconds_ms,rerank_seconds_ms
2882.1784,485.3671,38.5747,0.0066,2082.8061,2609.6631,0.0000,0.0000,2715.5212,Unavailable
5218.4194,832.0442,38.8916,0.0195,2434.2208,3580.3746,0.0000,0.0000,3619.0185,434.6148


## 7. Inspect failures and retries

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

This recorded reference run retained 1 failed timed attempts. Inspect their query, arm, duration and recorded backoff below; these costs remain included in total wall time.

No records are silently excluded from this benchmark.

In [8]:
view.show('failures')
view.show('attempts')


stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
generation,1083354,reranked,1,error,,7.9806,2.0011


stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
query_embedding,1083354,baseline,1,ok,200,0.3634,0.0000
stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
generation,1100471,reranked,1,ok,200,2.8335,0.0000
generation,1062919,reranked,1,ok,200,1.9257,0.0000
reranking,1084423,reranked,1,ok,200,0.4801,0.0000
query_embedding,1084423,reranked,1,ok,200,4.6494,0.0000
query_embedding,1100471,baseline,1,ok,200,0.9710,0.0000
reranking,106620,reranked,1,ok,200,0.4119,0.0000
generation,1084140,baseline,1,ok,200,2.5855,0.0000
query_embedding,106620,reranked,1,ok,200,0.3490,0.0000


## 8. The answer-scoring

**What:** score the 16 answers produced by the timed benchmark with the same libraries used in v3.

**Why:** compare latency with quality on the same question, answer and arm—not with an answer from another run.

Our preprocessing removes passage-ID citations while preserving answer text. Hugging Face SQuAD then applies its normalization and best-reference scoring across every retained reference. **Token F1 and normalized exact match use 0–100 percentages**. The custom-scorer comparison uses **0–1**.

RAGAS **Faithfulness** uses only that arm’s exact five benchmark passages. **Answer relevancy** uses the generated answer and question, with 2,560-dimensional Qwen evaluation embeddings. Their query instruction matches retrieval; evaluation results and caches are rebuilt separately.

RAGAS outcomes are reported for these Live-run answers with independently reported matching pair counts and any failures or missing values. Judge estimates are distinct from human correctness. Evaluation latency is outside the RAG path timings.


In [9]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))


```python
def score_squad_answers(inputs, metric, citation_pattern, legacy_citation_pattern):
    """Score each saved answer, keeping truncation and the old scorer comparison."""
    records = []
    for row in inputs:
        result = score_squad_answer(metric, row['answer'], row['reference_answers'], row['query_id'], citation_pattern)
        custom = score_custom_answer(row['answer'], row['reference_answers'], legacy_citation_pattern)
        for key, saved in [('token_f1', 'existing_token_f1_0_1'), ('normalized_em', 'existing_normalized_em_0_1')]:
            if saved in row and abs(custom[key] - row[saved]) > 1e-12:
                raise ValueError(f"Existing custom score changed: {row['query_id']} / {row['arm']} / {key}")
        records.append({key: row[key] for key in ['query_id', 'question', 'arm', 'path', 'answer', 'finish_reason', 'truncated',
                                                 'annotated_support_recall_at_5', 'existing_token_f1_0_1', 'existing_normalized_em_0_1']} |
                       {'squad_answer': remove_passage_citations(row['answer'], citation_pattern),
                        'reference_answers': json.dumps(row['reference_answers'], ensure_ascii=False),
                        'squad_exact_match_pct': result['exact_match'], 'squad_token_f1_pct': result['f1'],
                        'token_f1_delta_0_1': result['f1'] / 100 - custom['token_f1'],
                        'normalized_em_delta_0_1': result['exact_match'] / 100 - custom['normalized_em']})
    return pd.DataFrame(records)

```

## 9. Run answer scoring

**What:** compute SQuAD scores and RAGAS judgments for the benchmark’s 16 answers.

**Why:** compare latency and quality for the exact same timed answers.

Hugging Face `evaluate.load` and multiple-reference `metric.compute` provide local SQuAD scoring. The official SQuAD source is available locally.

RAGAS uses the real `llm_factory`, `Faithfulness` and `AnswerRelevancy` entry points. It requires `OPENROUTER_API_KEY` and logs prompts, settings, responses, failures, capped retries and timing. Evaluation happens outside the timed online paths. Inspect the implementations in the appendix.

In [10]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output("Show remaining latency scoring output — expand/collapse", preview_rows=1, preview_logs=True):
    score_run = await tutorial.score()
    if score_run is not None:
        display(score_run)
    view.show('judge_outcomes')


## 10. Inspect the latency–quality tradeoff

The numeric tables join each of eight within-question latency deltas with Token F1 for the exact timed answers. Positive latency delta means the reranked path took longer; positive F1 delta means more reference-word overlap.

If no plot is available, the numeric comparisons remain inspectable. If a plot is exported later, label it as eight pairs. No model calls are needed to render measured results.

Each populated subgroup has only two pairs. Inspect individual cases rather than treating subgroup bootstrap intervals as stable population evidence. Annotated-support Recall@5 stays separate on its 0–1 scale. Use the larger Experiment 3 for answer-quality analysis.


In [11]:
view.show('latency_quality')
view.show('scores')


query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds,question,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider
1046747,baseline,20261111T044228Z-219a4a03d428,5.2054,0.8792,0.1373,0.0000,4.1665,4.6810,0.0000,0.0000,4.8182,Unavailable,what is rx provigil u,baseline,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,False,1.0000,0.0404,0.0000,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,4.0404,0.0000,0.0000,1.0000,ok,0.6024,ok,control_all_support_already_present,False,95.0000,Nebius
query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds,question,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider
1046747,reranked,20261111T044228Z-219a4a03d428,4.1069,0.3744,0.0388,0.0000,3.2069,4.0139,0.0000,0.0000,4.0527,0.4700,what is rx provigil u,reranked,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_7110f1d3ec0ebf07c6b8][p_4db7d78a64f92cba650c].",stop,False,1.0000,0.0870,0.0000,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or sti

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,baseline,baseline,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,False,1.0000,0.0404,0.0000,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,4.0404,0.0000,0.0000,1.0000,ok,0.6024,ok,control_all_support_already_present,False,95.0000,Nebius,20261111T044228Z-219a4a03d428
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,reranked,reranked,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_7110f1d3ec0ebf07c6b8][p_4db7d78a64f92cba650c].",stop,False,1.0000,0.0870,0.0000,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,8.6957,-0.0000,0.0000,0.8750,ok,0.9284,ok,control_all_support_already_present,False,87.0000,Nebius,20261111T044228Z-219a4a03d428
106620,cost of repainting cabinets,reranked,reranked,"The average cost to repaint kitchen

subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,squad_token_f1_pct,2,10.3535,21.0145,10.6610,4.6552,16.6667
subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,faithfulness,2,1.0000,0.7708,-0.2292,-0.3333,-0.1250
control_all_support_already_present,reranked,answer_relevancy,2,0.7562,0.8859,0.1297,-0.0666,0.3260
control_all_support_already_present,reranked,answer_words,2,71.5000,57.0000,-14.5000,-21.0000,-8.0000
opportunity_gain,reranked,squad_token_f1_pct,2,16.3885,12.9014,-3.4871,-4.4297,-2.5445
opportunity_gain,reranked,faithfulness,2,1.0000,0.9500,-0.0500,-0.1000,0.0000
opportunity_gain,reranked,answer_relevancy,2,0.4401,0.4287,-0.0114,-0.0227,0.0000
opportunity_gain,reranked,answer_words,2,62.5000,74.0000,11.5000,2.0000,21.0000
opportunity_loss,reranked,squad_token_f1_pct,2,22.5951,21.8569,-0.7383,-2.6371,1.1605


## 11. Inspect one question with both paths

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

The example is **1046747, “what is rx provigil u”**. Both arms retain their exact Live-run contexts, answer records and timings. Inspect the displayed evidence rather than assuming the same passage selection or order.

Total wall time is **5.21 s → 4.11 s** and Token F1 is **4.04% → 8.70%**. F1 reflects reference-word overlap and answer length; a single pair does not establish a reliable latency effect.

Dataset passages can contain conflicting claims. Inspect this as an experimental output; human correctness review remains pending. Exact request bodies and response records remain in the Live-run artifacts.

In [12]:
view.show('walkthrough', query_id='1046747')


**1046747: what is rx provigil u**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,baseline,baseline,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,False,1.0000,0.0404,0.0000,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,4.0404,0.0000,0.0000,1.0000,ok,0.6024,ok,control_all_support_already_present,False,95.0000,Nebius,20261111T044228Z-219a4a03d428
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,reranked,reranked,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_7110f1d3ec0ebf07c6b8][p_4db7d78a64f92cba650c].",stop,False,1.0000,0.0870,0.0000,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,8.6957,-0.0000,0.0000,0.8750,ok,0.9284,ok,control_all_support_already_present,False,87.0000,Nebius,20261111T044228Z-219a4a03d428


## 12. Diagnostic membership

Neither IMEI **1053665** nor effervescent **1100870** belongs to the eight frozen latency questions. The Live-run membership output below checks both IDs. No answers, scores or timings from their separate walkthroughs are substituted into this benchmark.

In [13]:
view.show('diagnostic_membership')


{'1053665': False, '1100870': False}

## 13. Interpret the live-run findings

Compare paired latency changes with the scores of those exact timed answers. Report failures and retries from this run; all conclusions must follow from the displayed Live-run measurements.


In [14]:
view.show('latency')
view.show('scores')
tutorial.show()


query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,baseline,20261111T044228Z-219a4a03d428,5.2054,0.8792,0.1373,0.0000,4.1665,4.6810,0.0000,0.0000,4.8182,Unavailable
query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,reranked,20261111T044228Z-219a4a03d428,4.1069,0.3744,0.0388,0.0000,3.2069,4.0139,0.0000,0.0000,4.0527,0.4700
106620,reranked,20261111T044228Z-219a4a03d428,1.9253,0.3642,0.0390,0.0000,1.0644,1.8131,0.0000,0.0000,1.8522,0.4215
106620,baseline,20261111T044228Z-219a4a03d428,2.1702,0.3786,0.0377,0.0000,1.7190,2.0757,0.0000,0.0000,2.1134,Unavailable
1062919,baseline,20261111T044228Z-219a4a03d428,1.9093,0.2789,0.0374,0.0000,1.5628,1.8175,0.0000,0.0000,1.8548,Unavailable
1062919,reranked,20261111T044228Z-219a4a03d428,3.2594,0.8061,0.0385,0.0000,1.9382,3.1469,0.0000,0.0000,3.1853,0.4390
1084140,reranked,20261111T044228Z-219a4a03d428,3.2064,0.3117,0.0350,0.0000,2.2365,3.0996,0.0000,0.0000,3.1346,0.5837
1084140,baseline,20261111T044228Z-219a4a03d428,3.0119,0.3489,0.0359,0.0000,2.5947,2.9201,0.0000,0.0000,2.9561,Unavailable
1079518,baseline,20261111T044228Z-219a4a03d428,2.7525,0.7902,0.0399,0.0000,1.8922,2.6564,0.0000,0.0000,2.6962,Unavailable


total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
3.1846,0.5792,0.0670,0.0000,2.5066,2.9843,0.0000,0.0000,3.0513,Unavailable
5.9251,1.8112,0.0386,0.0000,3.4490,4.4170,0.9976,0.2501,4.4556,0.4498


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,baseline,baseline,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,False,1.0000,0.0404,0.0000,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,4.0404,0.0000,0.0000,1.0000,ok,0.6024,ok,control_all_support_already_present,False,95.0000,Nebius,20261111T044228Z-219a4a03d428
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1046747,what is rx provigil u,reranked,reranked,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_7110f1d3ec0ebf07c6b8][p_4db7d78a64f92cba650c].",stop,False,1.0000,0.0870,0.0000,"RX Provigil U refers to the prescription medication Provigil (modafinil), which is used to improve wakefulness in adults with excessive sleepiness due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is considered a nonstimulant or stimulant medication that works by altering brain chemicals (neurotransmitters) to increase alertness. It is not approved for use in children under 17, may interact with certain medications (including hormonal contraceptives), and is not recommended for long-term use due to potential habit-forming effects or liver issues .","[""Provigil is a stimulant that helps you feel awake.""]",0.0000,8.6957,-0.0000,0.0000,0.8750,ok,0.9284,ok,control_all_support_already_present,False,87.0000,Nebius,20261111T044228Z-219a4a03d428
106620,cost of repainting cabinets,reranked,reranked,"The average cost to repaint kitchen

subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,squad_token_f1_pct,2,10.3535,21.0145,10.6610,4.6552,16.6667
subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,faithfulness,2,1.0000,0.7708,-0.2292,-0.3333,-0.1250
control_all_support_already_present,reranked,answer_relevancy,2,0.7562,0.8859,0.1297,-0.0666,0.3260
control_all_support_already_present,reranked,answer_words,2,71.5000,57.0000,-14.5000,-21.0000,-8.0000
opportunity_gain,reranked,squad_token_f1_pct,2,16.3885,12.9014,-3.4871,-4.4297,-2.5445
opportunity_gain,reranked,faithfulness,2,1.0000,0.9500,-0.0500,-0.1000,0.0000
opportunity_gain,reranked,answer_relevancy,2,0.4401,0.4287,-0.0114,-0.0227,0.0000
opportunity_gain,reranked,answer_words,2,62.5000,74.0000,11.5000,2.0000,21.0000
opportunity_loss,reranked,squad_token_f1_pct,2,22.5951,21.8569,-0.7383,-2.6371,1.1605


**LIVE run `20261111T044228Z-219a4a03d428` — only this run’s answers, scores and timings.**

,squad_token_f1_pct,faithfulness,answer_relevancy,answer_words
arm,,,,
baseline,26.746062,0.982057,0.736539,52.375
reranked,29.556466,0.930208,0.763258,49.250


squad_token_f1_pct  \
subgroup                            arm                            
control_all_support_already_present baseline           10.353535   
                                    reranked           21.014493   
opportunity_gain                    baseline           16.388509   
                                    reranked           12.901440   
opportunity_loss                    baseline           22.595145   
                                    reranked           21.856867   
opportunity_unchanged               baseline           57.647059   
                                    reranked           62.453066   

                                              faithfulness  answer_relevancy  \
subgroup                            arm                                        
control_all_support_already_present baseline      1.000000          0.756205   
                                    reranked      0.770833          0.885910   
opportunity_gain                    baseline      1.000000          0.440062   
                                    reranked      0.950000          0.428698   
opportunity_loss                    baseline      0.928230          0.900756   
                                    reranked      1.000000          0.900756   
opportunity_unchanged               baseline      1.000000          0.849134   
                                    reranked      1.000000          0.837667   

                                              answer_words  
subgroup                            arm                     
control_all_support_already_present baseline          71.5  
                                    reranked          57.0  
opportunity_gain                    baseline          62.5  
                                    reranked          74.0  
opportunity_loss                    baseline          50.0  
                                    reranked          35.0  
opportunity_unchanged               baseline          25.5  
                                    reranked          31.0

arm       faithfulness_status
baseline  ok                     8
reranked  ok                     8
Name: metric_outcomes, dtype: int64

arm       answer_relevancy_status
baseline  ok                         8
reranked  ok                         8
Name: metric_outcomes, dtype: int64

,arm,answer,squad_token_f1_pct,faithfulness,answer_relevancy,answer_words


,query_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
arm,,,,,,,,,,,
baseline,956024.0,3.184612,0.579166,0.067027,0.000007,2.506638,2.984302,0.000000,0.000000,3.051336,NaN
reranked,956024.0,5.925111,1.811198,0.038580,0.000018,3.448977,4.417026,0.997577,0.250143,4.455624,0.449773


## Live benchmark execution

**What:** run fresh query embedding, retrieval, reranking and generation on the fixed latency sample.

**Why:** obtain measurements under your machine, network and endpoint conditions.

The live benchmark runs in section 3 by default. It requires the pinned dependencies, `OPENROUTER_API_KEY`, and the prepared corpus/index declared in `configs/tutorial_inputs.json`. Document embedding and index loading remain outside each timed path.

Measurements and per-attempt traces belong to the current run. Request caps and bounded retries apply. Scoring evaluates the benchmark’s own answers after the timed paths complete.

In [15]:
# The benchmark ran once in section 3. Inspect that run's measurements here.
# Start a new Live RUN_ID at setup to measure another run.
view.show('latency')


query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,baseline,20261111T044228Z-219a4a03d428,5.2054,0.8792,0.1373,0.0000,4.1665,4.6810,0.0000,0.0000,4.8182,Unavailable
query_id,arm,run_id,total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
1046747,reranked,20261111T044228Z-219a4a03d428,4.1069,0.3744,0.0388,0.0000,3.2069,4.0139,0.0000,0.0000,4.0527,0.4700
106620,reranked,20261111T044228Z-219a4a03d428,1.9253,0.3642,0.0390,0.0000,1.0644,1.8131,0.0000,0.0000,1.8522,0.4215
106620,baseline,20261111T044228Z-219a4a03d428,2.1702,0.3786,0.0377,0.0000,1.7190,2.0757,0.0000,0.0000,2.1134,Unavailable
1062919,baseline,20261111T044228Z-219a4a03d428,1.9093,0.2789,0.0374,0.0000,1.5628,1.8175,0.0000,0.0000,1.8548,Unavailable
1062919,reranked,20261111T044228Z-219a4a03d428,3.2594,0.8061,0.0385,0.0000,1.9382,3.1469,0.0000,0.0000,3.1853,0.4390
1084140,reranked,20261111T044228Z-219a4a03d428,3.2064,0.3117,0.0350,0.0000,2.2365,3.0996,0.0000,0.0000,3.1346,0.5837
1084140,baseline,20261111T044228Z-219a4a03d428,3.0119,0.3489,0.0359,0.0000,2.5947,2.9201,0.0000,0.0000,2.9561,Unavailable
1079518,baseline,20261111T044228Z-219a4a03d428,2.7525,0.7902,0.0399,0.0000,1.8922,2.6564,0.0000,0.0000,2.6962,Unavailable


total_end_to_end_seconds,query_embedding_seconds,retrieval_seconds,selection_prompt_seconds,generation_seconds,successful_request_seconds,failed_attempt_seconds,backoff_seconds,successful_path_seconds,rerank_seconds
3.1846,0.5792,0.0670,0.0000,2.5066,2.9843,0.0000,0.0000,3.0513,Unavailable
5.9251,1.8112,0.0386,0.0000,3.4490,4.4170,0.9976,0.2501,4.4556,0.4498


## Appendix: inspect the real pipeline and framework calls

**What:** read the actual benchmark path, hosted-request tracing and framework entry points.

**Why:** keep the tutorial’s API calls verifiable while setup and bulk execution stay in helpers. Displaying source performs no inference, retrieval, scoring or index loading.

In [16]:
display(Markdown("```python\n" + inspect.getsource(tutorial.benchmark) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(run_ragas_evaluation) + "\n```"))


```python
        return read_json(path)

```

```python
async def run_ragas_evaluation(inputs, source_directory, run_id, output_directory, settings,
                               allow_model_calls=False, retry_failures=False):
    """Opt in to judge calls; keep successes fixed and archive technical retries."""
    if not allow_model_calls:
        raise PermissionError('Judge calls are disabled. Use load_saved_judge_scores() for replay, or explicitly set allow_model_calls=True.')
    require_model_calls()
    SOURCE, OUT, CFG = Path(source_directory), Path(output_directory), settings
    rows = list(inputs)
    # Reject changed settings before replaying or extending an existing evaluation.
    manifest_path = OUT / 'judge_manifest.json'
    if manifest_path.exists():
        original = read(manifest_path)['settings']
        relevant = lambda values: {key: value for key, value in values.items()
                                  if key.startswith('judge_') or key.startswith('embedding_') or key in
                                  ['api_base', 'enable_thinking', 'relevancy_strictness', 'ragas_response_policy', 'query_instruction']}
        current = relevant(CFG)
        recorded = relevant(original)
        recorded.pop('judge_system_prompt', None)
        current.pop('judge_system_prompt', None)
        if recorded != current:
            raise ValueError('Judge/model settings changed. Use a separate explicit output directory; existing scores stay fixed.')
        for request_path in (OUT / 'judge_requests').glob('*.json'):
            request = read(request_path)
            if request['url'].endswith('/chat/completions'):
                if CFG['judge_system_prompt'] not in request['request_body']['messages'][0]['content']:
                    raise ValueError('Judge system prompt differs from saved requests. Use a separate output directory.')
                break
    existing = { (record['query_id'], record['arm'], record['metric']): record
                for record in [read(path) for path in (OUT / 'ragas_results').glob('*.json')] }
    if all((row['query_id'], row['arm'], metric) in existing and
           (existing[(row['query_id'], row['arm'], metric)]['status'] == 'ok' or
            (not retry_failures and existing[(row['query_id'], row['arm'], metric)]['status'] != 'started'))
           for row in rows for metric in CFG['metrics']):
        return load_saved_judge_scores(OUT, rows, CFG['metrics'], CFG)
    client, metrics = _make_components(SOURCE, run_id, OUT, CFG)
    semaphore=asyncio.Semaphore(CFG['concurrency'])
    async def evaluate(row,name,metric):
        path=OUT/'ragas_results'/f"{row['query_id']}-{row['arm']}-{name}.json";path.parent.mkdir(exist_ok=True)
        fingerprint=evaluation_fingerprint(row, name, CFG, importlib.metadata.version('ragas'))
        if path.exists():
            previous=read(path)
            if previous['fingerprint']!=fingerprint:
                raise ValueError(f'Changed evaluation input/settings: {path.name}. Use a separate output directory.')
            if previous['status']=='ok':return
            if not retry_failures:
                if previous['status']=='started':raise RuntimeError(f'Unfinished metric requires inspection: {path}')
                return
            archive=OUT/'ragas_attempt_history';archive.mkdir(exist_ok=True)
            prior=list(archive.glob(path.stem+'-*.json'))
            if len(prior)>=CFG['metric_max_attempts']-1:return
            if previous['status']=='started':
                previous.update(status='interrupted',score=None,error='Judge pass interrupted to apply rate-limit throttling',ended_utc=now())
            write(archive/f'{path.stem}-{len(prior)+1}.json',previous)
        async with semaphore:
            token=CONTEXT.set({'query_id':row['query_id'],'arm':row['arm'],'metric':name})
            result={'query_id':row['query_id'],'arm':row['arm'],'path':row['path'],'metric':name,'truncated':row['truncated'],
                'fingerprint':fingerprint,'judge_system_prompt':CFG['judge_system_prompt'],'status':'started','started_utc':now(),'context_passage_ids':row['context_passage_ids']}
            save_evaluation_attempt(path,result);start=time.perf_counter()
            try:
                kwargs={'user_input':row['question'],'response':row['answer']}
                if name=='faithfulness':kwargs['retrieved_contexts']=row['retrieved_contexts']
                score=await metric.ascore(**kwargs)
                value=float(score.value)
                if not math.isfinite(value):raise ValueError('RAGAS returned a non-finite score (e.g. no assessable statements)')
                result.update(status='ok',score=value)
            except asyncio.CancelledError:
                result.update(status='interrupted',score=None,error='Explicit judge evaluation interrupted before completion')
                raise
            except ModelBudgetExceeded:
                raise
            except Exception as e:
                result.update(status='error',score=None,error=f'{type(e).__name__}: {e}')
            finally:
                result.update(elapsed_seconds=time.perf_counter()-start,ended_utc=now());save_evaluation_attempt(path,result);CONTEXT.reset(token)
                print(f"{row['query_id']} {row['arm']} {name}: {result['status']} {result.get('score')} ({result['elapsed_seconds']:.1f}s)",flush=True)
    try:await asyncio.gather(*(evaluate(row,name,metric) for row in rows for name,metric in metrics.items()))
    finally:await client.close()
    return load_saved_judge_scores(OUT, rows, CFG['metrics'], CFG)

```

## Appendix: verify the frozen measurement and scoring inputs

**What:** validate the benchmark and scoring inputs with a local integrity check.

**Why:** verify the sample, prompts, answers, separate contexts, request/response traces, timing sums, retries and paired deltas.

Machine-specific asset paths are supplied through environment configuration. Notebook presentation hashes are informational because ordinary edits can change notebook bytes. Frozen parent data and configuration remain required dependencies.

See the adjacent README for setup, the file map and limitations.

The local integrity check displays its return value without requesting hosted model calls.

In [17]:
view.show('audit')


Check,Status
"Frozen inputs, question IDs and evidence contexts",Verified
